# Tools & Inbuilt Functions

Tools are reusable functions that LLM nodes can invoke during a run. The SDK
exposes three categories through `client.tools`:

1. **Inbuilt tools** — pre-packaged (math, string utilities); use as-is, no config.
2. **Configurable inbuilt tools** — pre-packaged but accept user-supplied config.
3. **Custom tools** — functions you author and manage with full CRUD.

This notebook lists the built-in catalogues, then creates, updates, and deletes a
custom tool. See also `../docs/guides/nodes_edges_tools.md`.

> **These notebooks are async-first.** They use `AsyncWorkflowClient` with top-level `await`,
> which runs directly in Jupyter (the setup cell calls `nest_asyncio.apply()`). Every method
> shown also exists on the synchronous `WorkflowClient` — just drop the `await`. See the
> [docs](../docs/README.md) for the sync surface. Notebook bodies stay 100% async — there is
> no per-notebook sync cell.

In [ ]:
import _bootstrap  # noqa: F401 - enables import interactly (no install needed)

import os, sys
import nest_asyncio
from dotenv import load_dotenv
from pathlib import Path

# This is required to run asyncio in Jupyter Notebook
nest_asyncio.apply()

# Get the current notebook directory and find the project root
current_dir = Path(os.getcwd())
project_root = current_dir
while project_root.parent != project_root:
    if (project_root / '.env').exists():
        break
    project_root = project_root.parent
else:
    project_root = current_dir
    for _ in range(5):
        if (project_root / 'pyproject.toml').exists():
            break
        project_root = project_root.parent

# Load the environment variables from .env in project root
env_path = project_root / '.env'
if env_path.exists():
    load_dotenv(dotenv_path=str(env_path), override=True)
    print(f"Loaded .env from: {env_path}")
else:
    print(f"Warning: .env file not found at {env_path}")

# Add the project root to sys.path so imports resolve
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))
    print(f"Added to Python path: {project_root}")
else:
    print(f"Project root already in Python path: {project_root}")

In [ ]:
# Interactly credentials are read from environment variables.
#
# Convenience defaults point at the dev "Workflow Illustrations" org; your shell
# environment always wins (setdefault only fills in what you have not set).
os.environ.setdefault("INTERACTLY_BASE_URL", "https://api-dev.interactly.ai/workflows")
os.environ.setdefault("INTERACTLY_TEAM_ID", "67458e762b7d3dc15aaea5b5")
os.environ.setdefault("INTERACTLY_USER_ID", "687b1a4f745c8e6806c98d91")

# The bearer token is a secret — never hardcode it in the notebook.
# Export it before launching Jupyter:  export INTERACTLY_API_KEY="…"
assert os.environ.get("INTERACTLY_API_KEY"), (
    "Set INTERACTLY_API_KEY in your environment before running this notebook."
)

#print(f"API KEY is: {os.getenv('INTERACTLY_API_KEY')}")
print(f"TEAM ID is: {os.getenv('INTERACTLY_TEAM_ID')}")
print(f"USER ID is: {os.getenv('INTERACTLY_USER_ID')}")
print(f"BASE URL is: {os.getenv('INTERACTLY_BASE_URL')}")

In [ ]:
import _bootstrap  # noqa: F401 - enables import interactly (no install needed)

import nest_asyncio

from interactly import AsyncWorkflowClient

# Required to run top-level `await` inside a Jupyter notebook
nest_asyncio.apply()

client = AsyncWorkflowClient()
print("Connected to", client._base_url)

## 1. Discover tool types and schemas

`types()` lists the custom tool type identifiers; `schema(tool_type)` returns the
JSON Schema (`config_schema`) describing the fields a tool of that type accepts.

In [ ]:
from typing import Dict, Any

tool_types = await client.tools.types()
print("Available custom tool types:", tool_types)

if tool_types:
    schema: Dict[str, Any] = await client.tools.schema(tool_types[0])
    print(f"\nSchema for {tool_types[0]!r}:")
    print(list(schema.get("config_schema", {}).get("properties", {}).keys()))

## 2. List inbuilt tools

Inbuilt tools require no configuration. Each entry carries `tool_id`, `name`,
`signature`, `args_schema`, and `category`.

In [ ]:
inbuilt = await client.tools.inbuilt()
print(f"{len(inbuilt)} inbuilt tools\n")
for tool in inbuilt[:10]:
    print(f"  [{tool.get('category', '?')}] {tool.get('tool_id')}: {tool.get('signature')}")

## 3. List configurable inbuilt tools

Unlike plain inbuilt tools, these expose a `config_schema` so you can supply
configuration when attaching them to a workflow. Each entry has `key`, `title`,
`registry_tool_id`, and `config_schema`.

In [ ]:
configurable = await client.tools.configurable_inbuilt()
print(f"{len(configurable)} configurable inbuilt tools\n")
for tool in configurable:
    print(f"  {tool.get('key')}: {tool.get('title')}")

In [ ]:
# Drill into one configurable tool's descriptor + config schema by its key
if configurable:
    key = configurable[0]["key"]
    descriptor = await client.tools.configurable_inbuilt_schema(key)
    print(f"Descriptor for {key!r}:")
    print("  title:", descriptor.get("title"))
    print("  registry_tool_id:", descriptor.get("registry_tool_id"))
    print("  config fields:", list(descriptor.get("config_schema", {}).get("properties", {}).keys()))

## 4. Create a custom tool

`create(tool_config=...)` leads with a typed `ToolConfig` object. Here we use
`InlinePythonToolConfig` — it carries a `name`, `description`, and the function
`code`. A `ToolConfig`-compatible dict is also accepted.

In [ ]:
from interactly.configs import InlinePythonToolConfig

# An inline-Python tool whose function looks a key up in an internal dictionary and returns the
# mapped value. To be *executable*, the config needs `signature` (what the tool does) and
# `args_schema` (its arguments), in addition to the `code`.
tool = await client.tools.create(
    tool_config=InlinePythonToolConfig(
        name="Capital Lookup",
        description="Return the capital city for a given country.",
        signature="Return the capital city of the given country.",
        args_schema={
            "type": "object",
            "properties": {"country": {"type": "string", "description": "Country name to look up"}},
            "required": ["country"],
        },
        code=(
            "def lookup_capital(country: str) -> str:\n"
            "    capitals = {\n"
            "        'france': 'Paris',\n"
            "        'japan': 'Tokyo',\n"
            "        'kenya': 'Nairobi',\n"
            "        'brazil': 'Brasilia',\n"
            "        'canada': 'Ottawa',\n"
            "    }\n"
            "    return capitals.get(country.strip().lower(), 'Unknown')"
        ),
    )
)

TOOL_ID = tool.id
print(f"Created tool id={TOOL_ID} name={tool.name!r}")

## 5. Execute the tool directly

`tools.execute(tool_id, args=...)` runs a **saved** tool server-side with the argument values you
pass and returns a typed `ToolExecuteResult` (`success`, `result`, `error`, `latency_ms`) — no
workflow required. `tools.execute_inline(tool_config=..., args=...)` does the same for an unsaved
config. Tool-level failures (bad args, exceptions, timeouts) come back as `success=False` with a
clean `error`, not an HTTP error.

In [ ]:
# Execute the SAVED tool by id, passing the lookup key directly.
result = await client.tools.execute(TOOL_ID, args={"country": "Japan"})
print(f"success={result.success}  result={result.result!r}  latency_ms={result.latency_ms}")
if result.error:
    print("error:", result.error)

In [ ]:
# You can also execute an INLINE (unsaved) tool config the same way, before persisting it.
inline_result = await client.tools.execute_inline(
    tool_config=InlinePythonToolConfig(
        name="Add",
        signature="Add two numbers and return the sum.",
        args_schema={
            "type": "object",
            "properties": {"a": {"type": "number"}, "b": {"type": "number"}},
            "required": ["a", "b"],
        },
        code="def add(a: float, b: float) -> float:\n    return a + b",
    ),
    args={"a": 2, "b": 40},
)
print(f"inline: success={inline_result.success}  result={inline_result.result}")

## 6. Read it back and list custom tools

`list()` is paginated — iterate the page directly or call `list_all()`.

In [ ]:
fetched = await client.tools.get(TOOL_ID)
print("Fetched:", fetched.id, fetched.name)

page = await client.tools.list(page=1, size=20, search="capital")
async for t in page:
    print("  ", t.id, t.name)

## 7. Update the custom tool

`update(tool_id, tool_config=...)` sends only the fields you pass. Here we tweak
the description and the function body.

In [ ]:
updated = await client.tools.update(
    TOOL_ID,
    tool_config=InlinePythonToolConfig(
        description="Return the capital city for a given country (expanded coverage).",
        code=(
            "def lookup_capital(country: str) -> str:\n"
            "    capitals = {\n"
            "        'france': 'Paris',\n"
            "        'japan': 'Tokyo',\n"
            "        'kenya': 'Nairobi',\n"
            "        'brazil': 'Brasilia',\n"
            "        'canada': 'Ottawa',\n"
            "        'india': 'New Delhi',\n"
            "    }\n"
            "    return capitals.get(country.strip().lower(), 'Unknown')"
        ),
    ),
)
print("Updated description:", updated.description)

## Moving a tool between teams or environments

`export()` produces a portable bundle; `import_bundle()` recreates it elsewhere.

Two things the server insists on, both worth knowing before the call fails:

- **Secrets are redacted on export**, never included. The bundle records which fields were dropped;
  re-supply them by dotted path via `secret_overrides`, e.g.
  `{"api_headers.Authorization": "Bearer …"}`.
- **`inline_python` bundles require `confirm_executable=True`.** Such a bundle carries code that
  will run in *your* team's context, so it is refused unless you say so explicitly.

`clear_unresolved_refs` (default `True`) strips team-scoped references — knowledge-base ids and the
like — that cannot resolve in the importing team, rather than importing a tool that silently points
at another team's resources.

In [ ]:
bundle = await client.tools.export(TOOL_ID)
print("bundle keys:", sorted(bundle.keys())[:10])

# Round-trip it back into this same team under a new name.
imported = await client.tools.import_bundle(
    bundle,
    name_override="Imported copy (notebook 04)",
    # This bundle is an inline_python tool, so the server refuses it without this flag.
    confirm_executable=True,
)
print(f"\nimported: {imported.id}  {imported.name!r}")

await client.tools.delete(imported.id)
print("cleaned up the imported copy")

## 8. Cleanup

Delete the custom tool we created so we leave no residue.

In [ ]:
await client.tools.delete(TOOL_ID)
print(f"Deleted tool {TOOL_ID}.")

await client.close()

## See also

- Guide: [`../docs/guides/nodes_edges_tools.md`](../docs/guides/nodes_edges_tools.md)
- [`08_full_crud_walkthrough.ipynb`](08_full_crud_walkthrough.ipynb) — end-to-end nodes, edges, and versions CRUD
- [`02_interactive_workflow.ipynb`](02_interactive_workflow.ipynb) — building a workflow graph and chatting with it
